# 6.4 Other Frameworks to ONNX — Apply Notebook

## Objective

Convert models from **XGBoost**, **LightGBM**, and other gradient-boosting
frameworks to ONNX.  Validate predictions, compare sizes, and benchmark
inference speed.

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Survey available converters | Runtime availability checks |
| 2 | Convert an XGBoost model | `onnxmltools` / `skl2onnx` |
| 3 | Convert a LightGBM model | `onnxmltools.convert_lightgbm` |
| 4 | Validation checklist for converted models | Systematic correctness test |
| 5 | ONNX model inspection after conversion | Op types, tree structure |
| 6 | Performance comparison | Framework vs ORT latency |
| 7 | Size comparison across frameworks | Serialized model sizes |
| 8 | **Challenge:** build a converter registry | Dispatch pattern |

```
pip install onnx onnxruntime onnxmltools skl2onnx scikit-learn numpy
pip install xgboost lightgbm  # optional
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, tempfile, warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

import onnx
from onnx import checker
import onnxruntime as ort

# ── Availability checks ───────────────────────────────────────────────
HAS_XGBOOST = False
try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    pass

HAS_LIGHTGBM = False
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    pass

HAS_ONNXMLTOOLS = False
try:
    import onnxmltools
    HAS_ONNXMLTOOLS = True
except ImportError:
    pass

HAS_SKL2ONNX = False
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    HAS_SKL2ONNX = True
except ImportError:
    pass

print(f"ONNX         : {onnx.__version__}")
print(f"ORT          : {ort.__version__}")
print(f"XGBoost      : {xgb.__version__ if HAS_XGBOOST else 'NOT INSTALLED'}")
print(f"LightGBM     : {lgb.__version__ if HAS_LIGHTGBM else 'NOT INSTALLED'}")
print(f"onnxmltools  : {onnxmltools.__version__ if HAS_ONNXMLTOOLS else 'NOT INSTALLED'}")
print(f"skl2onnx     : {'available' if HAS_SKL2ONNX else 'NOT INSTALLED'}")

## Exercise 1 — Survey Available Converters

Before converting a model, check which frameworks and converters are
installed.  This avoids cryptic import errors at conversion time.

In [ ]:
converters = {
    "XGBoost → ONNX": HAS_XGBOOST and HAS_ONNXMLTOOLS,
    "LightGBM → ONNX": HAS_LIGHTGBM and HAS_ONNXMLTOOLS,
    "sklearn → ONNX": HAS_SKL2ONNX,
}

print("Converter availability:")
print("─" * 40)
for name, avail in converters.items():
    status = "READY" if avail else "MISSING DEPS"
    print(f"  {name:<24} {status}")

n_ready = sum(converters.values())
print(f"\n{n_ready}/{len(converters)} converters ready.")
if n_ready < len(converters):
    print("Install missing packages:  pip install xgboost lightgbm onnxmltools skl2onnx")

## Shared Dataset

We generate a single classification dataset used across all exercises so
comparisons are apples-to-apples.

In [ ]:
X_all, y_all = make_classification(
    n_samples=1000, n_features=10, n_informative=8,
    n_classes=3, random_state=42,
)
X_all = X_all.astype(np.float32)
X_train, X_test = X_all[:800], X_all[800:]
y_train, y_test = y_all[:800], y_all[800:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Classes: {np.unique(y_train)}")

## Exercise 2 — Convert an XGBoost Model

XGBoost builds boosted trees using a second-order Taylor expansion of the loss:

$$\mathcal{L}^{(t)} \approx \sum_{i} \left[ g_i f_t(\mathbf{x}_i) + \tfrac{1}{2} h_i f_t^2(\mathbf{x}_i) \right] + \Omega(f_t)$$

where $g_i, h_i$ are the first and second derivatives of the loss w.r.t. the
prediction.  `onnxmltools.convert_xgboost` converts the trained booster.

In [ ]:
if HAS_XGBOOST and HAS_ONNXMLTOOLS:
    from onnxmltools.convert import convert_xgboost
    from onnxmltools.convert.common.data_types import FloatTensorType as FTT

    xgb_model = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        use_label_encoder=False, eval_metric="mlogloss", random_state=0,
    )
    xgb_model.fit(X_train, y_train)
    xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
    print(f"XGBoost test accuracy: {xgb_acc:.4f}")

    onnx_xgb = convert_xgboost(
        xgb_model,
        initial_types=[("X", FTT([None, X_train.shape[1]]))],
        target_opset=17,
    )
    checker.check_model(onnx_xgb)
    print(f"ONNX nodes: {len(onnx_xgb.graph.node)}")
    print(f"Ops: {sorted(set(n.op_type for n in onnx_xgb.graph.node))}")

    sess = ort.InferenceSession(onnx_xgb.SerializeToString(), providers=["CPUExecutionProvider"])
    ort_pred = sess.run(None, {"X": X_test})[0]
    match = np.mean(xgb_model.predict(X_test) == ort_pred)
    print(f"Label match rate: {match:.4f}")
    assert match > 0.99
    print("XGBoost conversion verified ✓")
else:
    print("⚠ XGBoost or onnxmltools not installed — skipping.")
    print("  pip install xgboost onnxmltools")
    onnx_xgb = None

## Exercise 3 — Convert a LightGBM Model

LightGBM uses histogram-based splitting and leaf-wise growth.  The
conversion path is similar to XGBoost via `onnxmltools.convert_lightgbm`.

In [ ]:
if HAS_LIGHTGBM and HAS_ONNXMLTOOLS:
    from onnxmltools.convert import convert_lightgbm
    from onnxmltools.convert.common.data_types import FloatTensorType as FTT

    lgb_model = lgb.LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        num_leaves=31, random_state=0, verbose=-1,
    )
    lgb_model.fit(X_train, y_train)
    lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test))
    print(f"LightGBM test accuracy: {lgb_acc:.4f}")

    onnx_lgb = convert_lightgbm(
        lgb_model,
        initial_types=[("X", FTT([None, X_train.shape[1]]))],
        target_opset=17,
    )
    checker.check_model(onnx_lgb)
    print(f"ONNX nodes: {len(onnx_lgb.graph.node)}")
    print(f"Ops: {sorted(set(n.op_type for n in onnx_lgb.graph.node))}")

    sess = ort.InferenceSession(onnx_lgb.SerializeToString(), providers=["CPUExecutionProvider"])
    ort_pred = sess.run(None, {"X": X_test})[0]
    match = np.mean(lgb_model.predict(X_test) == ort_pred)
    print(f"Label match rate: {match:.4f}")
    assert match > 0.99
    print("LightGBM conversion verified ✓")
else:
    print("⚠ LightGBM or onnxmltools not installed — skipping.")
    print("  pip install lightgbm onnxmltools")
    onnx_lgb = None

## Exercise 4 — Validation Checklist for Converted Models

A systematic checklist that every converted model should pass before
deployment:

1. `onnx.checker.check_model` — structural validity
2. Label parity — same predicted classes
3. Probability parity — same confidence scores
4. Dynamic batch — accepts varying batch sizes
5. Edge cases — zeros, large values, single sample

In [ ]:
def validation_checklist(name, onnx_proto, predict_fn, X_sample):
    """Run a standardised validation suite on a converted ONNX model."""
    results = []

    # 1. Checker
    try:
        checker.check_model(onnx_proto)
        results.append(("onnx.checker", "PASS"))
    except Exception as e:
        results.append(("onnx.checker", f"FAIL: {e}"))

    sess = ort.InferenceSession(
        onnx_proto.SerializeToString(), providers=["CPUExecutionProvider"]
    )
    in_name = sess.get_inputs()[0].name

    # 2. Label parity
    native_pred = predict_fn(X_sample)
    ort_pred = sess.run(None, {in_name: X_sample})[0]
    label_match = np.mean(native_pred == ort_pred)
    results.append(("Label parity", f"{label_match:.4f}" + (" PASS" if label_match > 0.99 else " FAIL")))

    # 3. Dynamic batch
    batch_ok = True
    for bs in [1, 5, 50]:
        try:
            x = np.random.randn(bs, X_sample.shape[1]).astype(np.float32)
            out = sess.run(None, {in_name: x})[0]
            if out.shape[0] != bs:
                batch_ok = False
        except Exception:
            batch_ok = False
    results.append(("Dynamic batch", "PASS" if batch_ok else "FAIL"))

    # 4. Edge cases
    edge_ok = True
    for label, x in [("zeros", np.zeros_like(X_sample[:2])),
                     ("large", X_sample[:2] * 1000)]:
        try:
            sess.run(None, {in_name: x})
        except Exception:
            edge_ok = False
    results.append(("Edge cases", "PASS" if edge_ok else "FAIL"))

    print(f"\n  Validation: {name}")
    print(f"  {'─'*40}")
    for check, status in results:
        print(f"  {check:<24} {status}")
    return all("PASS" in s for _, s in results)


# Run checklist on all available models
if HAS_XGBOOST and onnx_xgb is not None:
    assert validation_checklist("XGBoost", onnx_xgb, xgb_model.predict, X_test)

if HAS_LIGHTGBM and onnx_lgb is not None:
    assert validation_checklist("LightGBM", onnx_lgb, lgb_model.predict, X_test)

if not (HAS_XGBOOST or HAS_LIGHTGBM):
    print("No framework models available for validation. Install xgboost or lightgbm.")

## Exercise 5 — ONNX Model Inspection After Conversion

Boosted-tree models convert to `TreeEnsembleClassifier` (an `ai.onnx.ml` op)
which encodes the entire tree structure in its attributes.  Let's compare
the internal graph structure across frameworks.

In [ ]:
from collections import Counter

models_to_inspect = []
if onnx_xgb is not None:
    models_to_inspect.append(("XGBoost", onnx_xgb))
if onnx_lgb is not None:
    models_to_inspect.append(("LightGBM", onnx_lgb))

if not models_to_inspect:
    print("No models to inspect. Install xgboost or lightgbm + onnxmltools.")
else:
    for name, proto in models_to_inspect:
        g = proto.graph
        init_names = {i.name for i in g.initializer}

        print(f"\n{'═'*50}")
        print(f"  {name} ONNX Graph")
        print(f"{'═'*50}")

        print(f"  Opset: {proto.opset_import[0].version}")
        for oi in proto.opset_import:
            if oi.domain:
                print(f"  Domain: {oi.domain} v{oi.version}")

        print(f"\n  Inputs:")
        for inp in g.input:
            if inp.name not in init_names:
                dims = [d.dim_param or str(d.dim_value) for d in inp.type.tensor_type.shape.dim]
                print(f"    {inp.name}: [{', '.join(dims)}]")

        print(f"  Outputs:")
        for o in g.output:
            print(f"    {o.name}")

        counts = Counter(n.op_type for n in g.node)
        print(f"\n  Nodes ({len(g.node)} total):")
        for op, c in counts.most_common():
            print(f"    {op:<30} {c}")

        print(f"  Serialized: {len(proto.SerializeToString())/1024:.1f} KB")

## Exercise 6 — Performance Comparison

Benchmark native framework `predict` vs ORT inference.

$$\text{speedup} = \frac{t_{\text{native}}}{t_{\text{ORT}}}$$

In [ ]:
def bench(fn, warmup=20, iters=100):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    arr = np.array(times) * 1000
    return {"median": np.median(arr), "p95": np.percentile(arr, 95)}


benchmarks = []

if HAS_XGBOOST and onnx_xgb is not None:
    sess_xgb = ort.InferenceSession(onnx_xgb.SerializeToString(), providers=["CPUExecutionProvider"])
    in_name = sess_xgb.get_inputs()[0].name
    native = bench(lambda: xgb_model.predict(X_test))
    ort_b  = bench(lambda: sess_xgb.run(None, {in_name: X_test}))
    benchmarks.append(("XGBoost", native["median"], ort_b["median"]))

if HAS_LIGHTGBM and onnx_lgb is not None:
    sess_lgb = ort.InferenceSession(onnx_lgb.SerializeToString(), providers=["CPUExecutionProvider"])
    in_name = sess_lgb.get_inputs()[0].name
    native = bench(lambda: lgb_model.predict(X_test))
    ort_b  = bench(lambda: sess_lgb.run(None, {in_name: X_test}))
    benchmarks.append(("LightGBM", native["median"], ort_b["median"]))

if benchmarks:
    print(f"{'Framework':<12} {'Native (ms)':>12} {'ORT (ms)':>12} {'Speedup':>10}")
    print("─" * 50)
    for name, nat_ms, ort_ms in benchmarks:
        speedup = nat_ms / ort_ms
        print(f"{name:<12} {nat_ms:>10.3f} ms {ort_ms:>10.3f} ms {speedup:>9.2f}x")
    print("\n(Speedup > 1 means ORT is faster)")
else:
    print("No frameworks available for benchmarking.")

## Exercise 7 — Size Comparison Across Frameworks

Compare the serialized sizes of native format vs ONNX for each framework.

In [ ]:
import pickle

size_data = []

if HAS_XGBOOST and onnx_xgb is not None:
    with tempfile.TemporaryDirectory() as tmp:
        native_path = os.path.join(tmp, "xgb.json")
        xgb_model.save_model(native_path)
        native_kb = os.path.getsize(native_path) / 1024
    onnx_kb = len(onnx_xgb.SerializeToString()) / 1024
    size_data.append(("XGBoost", native_kb, onnx_kb))

if HAS_LIGHTGBM and onnx_lgb is not None:
    with tempfile.TemporaryDirectory() as tmp:
        native_path = os.path.join(tmp, "lgb.txt")
        lgb_model.booster_.save_model(native_path)
        native_kb = os.path.getsize(native_path) / 1024
    onnx_kb = len(onnx_lgb.SerializeToString()) / 1024
    size_data.append(("LightGBM", native_kb, onnx_kb))

if size_data:
    print(f"{'Framework':<12} {'Native (KB)':>12} {'ONNX (KB)':>12} {'Ratio':>8}")
    print("─" * 48)
    for name, nat, onx in size_data:
        ratio = nat / onx if onx > 0 else float("inf")
        print(f"{name:<12} {nat:>10.1f} {onx:>10.1f} {ratio:>7.2f}x")
else:
    print("No models available for size comparison.")

## Exercise 8 — Challenge: Build a Converter Registry

Create a **registry** that dispatches conversion based on the model type.
This pattern is useful in ML platforms where users submit arbitrary models
and the platform auto-converts to ONNX.

```
registry.convert(any_model, X_sample)  →  onnx.ModelProto
```

In [ ]:
class ConverterRegistry:
    """Dispatch ONNX conversion based on model type."""

    def __init__(self):
        self._converters = {}

    def register(self, model_class, converter_fn):
        self._converters[model_class] = converter_fn

    def convert(self, model, X_sample, target_opset=17):
        for cls, fn in self._converters.items():
            if isinstance(model, cls):
                return fn(model, X_sample, target_opset)
        raise TypeError(f"No converter registered for {type(model).__name__}")

    def list_supported(self):
        return [cls.__name__ for cls in self._converters]


registry = ConverterRegistry()

# Register sklearn converter
if HAS_SKL2ONNX:
    from sklearn.base import BaseEstimator

    def _convert_sklearn(model, X_sample, opset):
        initial_type = [("X", FloatTensorType([None, X_sample.shape[1]]))]
        return convert_sklearn(model, initial_types=initial_type, target_opset=opset)

    registry.register(BaseEstimator, _convert_sklearn)

# Register XGBoost converter
if HAS_XGBOOST and HAS_ONNXMLTOOLS:
    from onnxmltools.convert import convert_xgboost
    from onnxmltools.convert.common.data_types import FloatTensorType as FTT

    def _convert_xgb(model, X_sample, opset):
        return convert_xgboost(model, initial_types=[("X", FTT([None, X_sample.shape[1]]))], target_opset=opset)

    registry.register(xgb.XGBClassifier, _convert_xgb)

# Register LightGBM converter
if HAS_LIGHTGBM and HAS_ONNXMLTOOLS:
    from onnxmltools.convert import convert_lightgbm
    from onnxmltools.convert.common.data_types import FloatTensorType as FTT

    def _convert_lgb(model, X_sample, opset):
        return convert_lightgbm(model, initial_types=[("X", FTT([None, X_sample.shape[1]]))], target_opset=opset)

    registry.register(lgb.LGBMClassifier, _convert_lgb)

print(f"Supported model types: {registry.list_supported()}")

# Test the registry with each available model
test_models = []
if HAS_XGBOOST:
    test_models.append(("XGBoost", xgb_model))
if HAS_LIGHTGBM:
    test_models.append(("LightGBM", lgb_model))

if HAS_SKL2ONNX:
    from sklearn.linear_model import LogisticRegression
    sk_lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    test_models.append(("sklearn LR", sk_lr))

for name, model in test_models:
    try:
        proto = registry.convert(model, X_test)
        checker.check_model(proto)
        sess = ort.InferenceSession(proto.SerializeToString(), providers=["CPUExecutionProvider"])
        out = sess.run(None, {sess.get_inputs()[0].name: X_test[:5]})[0]
        print(f"  {name:<14} → converted OK, output shape: {out.shape}")
    except Exception as e:
        print(f"  {name:<14} → FAILED: {e}")

if not test_models:
    print("No models available to test the registry.")

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| Availability survey | Always check framework + converter presence before converting |
| XGBoost | `onnxmltools.convert_xgboost` for boosted trees |
| LightGBM | `onnxmltools.convert_lightgbm` for histogram-based boosting |
| Validation | Systematic checklist: checker → labels → probabilities → batch → edge cases |
| Inspection | Tree models use `TreeEnsembleClassifier` from `ai.onnx.ml` domain |
| Performance | ORT's tree kernel can be faster than native Python bindings |
| Size | ONNX protobuf and native format sizes vary by framework |
| Registry | A dispatch pattern automates conversion in ML platforms |